<a href="https://colab.research.google.com/github/deadex-ng/arena/blob/main/chapter0_fundamentals/exercises/part2_cnns/BatchNorm2d.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
class BatchNorm2d(nn.Module):
    def __init__(self, num_features: int, eps: float = 1e-5):
        super().__init__()
        self.num_features = num_features
        self.eps = eps

        # Learnable parameters
        self.gamma = nn.Parameter(torch.ones(1, num_features, 1, 1))
        self.beta  = nn.Parameter(torch.zeros(1, num_features, 1, 1))

    def forward(self, x: Tensor) -> Tensor:
        mean = x.mean(dim=(0, 2, 3), keepdim=True)
        var  = x.var(dim=(0, 2, 3), unbiased=False, keepdim=True)

        x_hat = (x - mean) / torch.sqrt(var + self.eps)
        y = self.gamma * x_hat + self.beta
        return y


In [1]:
import torch
import torch.nn as nn
from torch import Tensor

class SimpleCNN(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv = nn.Conv2d(
        in_channels=3,
        out_channels=5,
        kernel_size=3,
        padding=1
    )

    self.bn = BatchNorm2d(num_features=5)
    self.relu = nn.ReLU()

  def forward(self, x: Tensor) -> Tensor:
    x = self.conv(x)
    x = self.bn(x)
    x = self.relu(x)
    return x

In [3]:
model = SimpleCNN()
x = torch.randn(2, 3, 4, 4)
output = model(x)
print(output.shape)

torch.Size([2, 5, 4, 4])


# CNN Forward Pass (With BatchNorm)

## Convolution
Conv output → wild values

## Batch Normalization
- Subtract mean → center at 0  
- Divide by std → normalize spread  
- Multiply γ → adjust feature strength  
- Add β → adjust feature center  

## Activation
ReLU → clean signal

## Input
Example input image batch:

- Shape: `(2, 3, 4, 4)`
- Meaning:
  - 2 images
  - 3 channels (RGB)
  - 4 × 4 pixels

---

## Convolution
```text
Input  → Conv2d(3 → 5)
Output → shape (2, 5, 4, 4)
```

## Batch Normalization
### Compute Statistics (per channel)

```text
Mean shape: (1, 5, 1, 1)

Variance shape: (1, 5, 1, 1)

```
Computed over:

- all images in the batch

- all spatial positions

## Normalize

- Subtract mean → center at 0

- Divide by std → normalize spread

After normalization:

- mean ≈ 0

- variance ≈ 1

## Learnable Adjustment

- Multiply γ → adjust feature strength

- Add β → adjust feature center

Initially:
```text
- γ = 1

- β = 0
```
## Activation

ReLU → clean signal
(negative values → 0, positive values pass through)

## Final Output

- Shape: (2, 5, 4, 4)

- Well-scaled features ready for the next layer